# Pruebas controladas del generador
Comparte las funciones de `generar_reportes.py`. Consulta un rango corto, revisa datos y cobertura, y genera sólo lo que elijas. La extracción ocurre una vez; puedes volver a escribir libros usando `resultado` sin consultar de nuevo.

Los diarios y los intervalos se solapan deliberadamente: son tablas distintas para análisis distintos. No sumar ambas entre sí. En Workloads diario, `requerido_horas` es dotación por duración; cada TMO usa el volumen de su propia variante. En Estados, las duraciones son horas decimales, sin crear intervalos para agentes ausentes.


In [ ]:
from datetime import date
from pathlib import Path
import polars as pl
from IPython.display import display
from analytics.flujo_reportes import cargar_datos, tablas_reportes

actual = Path.cwd().resolve()
BASE_DIR = next((p for p in (actual, *actual.parents)
                 if (p / "src" / "analytics").is_dir() and (p / "pyproject.toml").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Abrir el notebook dentro del proyecto.")
from analytics.flujo_reportes import listar_cortes, generar_reportes

DESDE = date(2026, 10, 8)
HASTA = date(2026, 10, 12)
DESDE_WORKLOADS = DESDE
PLANTILLA = BASE_DIR / "archivos" / "plantilla_reportes_base.xlsx"
CARPETA_PRUEBA = BASE_DIR / "archivos" / "pruebas_reportes"


In [ ]:
resultado = cargar_datos(DESDE, HASTA, desde_workloads=DESDE_WORKLOADS,
                         semanas_intervalos=8, base_dir=BASE_DIR)
disponibles = listar_cortes(resultado)


In [ ]:
FUENTE = "workloads_diario"  # Cualquiera de las seis fuentes.
df = resultado.datos[FUENTE]
display(df.head(20))
display(pl.DataFrame({"columna": list(df.schema),
                      "tipo": [str(t) for t in df.schema.values()],
                      "nulos": list(df.null_count().row(0))}))
display(resultado.workloads.alertas)
display(resultado.workloads.cobertura_fuentes)


In [ ]:
# Primero sólo el libro general. Activar los cortes y elegir nombres después.
GENERAR_CLIENTES = False
GENERAR_JEFES = False
CLIENTES = None  # Ejemplo: ["Mutual"]. None genera todos los de ese criterio.
JEFES = None    # Nombres exactos de disponibles["jefe_negocio"].

resumen = generar_reportes(
    resultado, PLANTILLA,
    salida_general=CARPETA_PRUEBA / "reporte_general.xlsx",
    salida_clientes=CARPETA_PRUEBA / "clientes" if GENERAR_CLIENTES else None,
    salida_jefes=CARPETA_PRUEBA / "jefes" if GENERAR_JEFES else None,
    clientes=CLIENTES, jefes=JEFES,
    ruta_resumen=CARPETA_PRUEBA / "resumen_generacion.json",
)
print(resumen["estado"], len(resumen["archivos"]), "libros verificados")
